In [2]:
import torch
import lightning.pytorch as pl

In [3]:
# x = torch.randn((3, 3))
x = torch.randint(-15, 15, (3,3))
print(x)

tensor([[  9,  -2,   7],
        [-10,  -2,  -2],
        [ -3,  14, -13]])


## BReLU 기본형

In [4]:
class BReLU(pl.LightningModule):
    def __init__(self):
        super(BReLU, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        return x * epsilon.to(self.device)

In [5]:
brelu = BReLU()
brelu(x)

tensor([[ 9., -0.,  7.],
        [-0., -2., -0.],
        [-0., 14., -0.]])

In [6]:
print(brelu)

BReLU()


## Leaky BReLU

In [7]:
class Leaky_BReLU(pl.LightningModule):
    def __init__(self):
        super(Leaky_BReLU, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        epsilon = torch.where(epsilon==0, 0.1, 1.0)
        
        return x * epsilon.to(self.device)

In [8]:
leaky = Leaky_BReLU()
leaky(x)

tensor([[ 9.0000, -0.2000,  7.0000],
        [-1.0000, -0.2000, -0.2000],
        [-0.3000, 14.0000, -1.3000]])

In [9]:
print(leaky)

Leaky_BReLU()


## 점근 속도를 조절하는 BReLU

In [10]:
class VariableBReLU(pl.LightningModule):
    def __init__(self, alpha=1):
        super(VariableBReLU, self).__init__()
        self.alpha = alpha
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=self.alpha * x).sample()
        
        return x * epsilon.to(self.device)
    
    def extra_repr(self) -> str:
        return 'alpha={}'.format(self.alpha)

In [11]:
def mv(**kwargs):
    if 'alpha' in list(kwargs.keys()):
        return VariableBReLU(alpha=kwargs.get('alpha'))

mv(alpha=.1)

VariableBReLU(alpha=0.1)

In [12]:
alpha = 0.1
vb = VariableBReLU(alpha=alpha)
vb(x)

tensor([[  9.,  -0.,   0.],
        [-10.,  -0.,  -2.],
        [ -3.,   0.,  -0.]])

In [13]:
print(vb)

VariableBReLU(alpha=0.1)


## 결과값을 0아니면 1로 만들기

In [14]:
class MeasureAct(pl.LightningModule):
    def __init__(self):
        super(MeasureAct, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        epsilon = epsilon * (1/(x + 1e-16))
        
        return x * epsilon.to(self.device)

In [15]:
ma = MeasureAct()
ma(x)

tensor([[1., 1., 1.],
        [0., 0., 0.],
        [0., 1., 0.]])

## 일부 원소만 0 아니면 1로 만들기

In [16]:
class SomeMeasureAct(pl.LightningModule):
    def __init__(self):
        super(SomeMeasureAct, self).__init__()
        
    def forward(self, x):
        b = torch.distributions.bernoulli.Bernoulli(logits=x)
        b1 = b.sample()
        b2 = b.sample()

        epsilon = b1*b2*(1/(x+1e-16)) + torch.logical_xor(b1, b2)
        
        return x * epsilon

In [17]:
sma = SomeMeasureAct()
sma(x)

tensor([[1., -0., 1.],
        [-0., -0., -0.],
        [-0., 1., -0.]])

## 확률적으로 ReLU, BReLU를 적용하기

In [18]:
class SARB(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(SARB, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        mu = x.to(torch.float32).mean()
        b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        if b == 0:
            return self.zero(x)
        
        else:
            return self.one(x)
        
    # def extra_repr(self) -> str:
    #     return 'zero={}, one={}'.format(self.zero, self.one)
        

In [19]:
class SARBSC(pl.LightningModule):                         #Stochastic Activation Relu or Brelu(확률적으로 Relu나 BRelu를 적용)
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(SARBSC, self).__init__() 
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        mu = x.to(torch.float32).mean()
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        # b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        b = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        if b == 0:
            return self.zero(x)
        
        else:
            return self.one(x)

In [ ]:
class BatchSC(pl.LightningModule):                         #Stochastic Activation Relu or Brelu(확률적으로 Relu나 BRelu를 적용)
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(BatchSC, self).__init__() 
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        tmp = []
        mu = x.to(torch.float32).mean(dim=(1,2,3), keepdim=False)
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        # b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        switch = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        for b, batch in zip(switch, x):
            if b == 0:
                tmp.append(self.zero(batch))
            
            else:
                tmp.append(self.one(batch))

In [20]:
sarb = SARB()
sarb(x)

tensor([[ 9,  0,  7],
        [ 0,  0,  0],
        [ 0, 14,  0]])

In [21]:
sarb

SARB(
  (zero): BReLU()
  (one): ReLU(inplace=True)
)

배치별로 다른 활성화함수 적용 또는 채널별

In [74]:
tx = torch.randint(-5, 5, (4, 3, 2, 2)).to(torch.float32)
# ty = []
# for i in tx:
#     print(i-1)
#     ty.append(i-1)
# ty = torch.stack(ty)
print(tx)
# print(ty)
mu = tx.mean(dim=(1, 2, 3), keepdim=False)
print(mu.shape)
# print(torch.distributions.bernoulli.Bernoulli(logits=tx.mean(dim=(1,2, 3), keepdim=True)).sample().expand_as(tx))
p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
p = torch.where(p < 0, 0, p)
p = torch.where(p > 1, 1, p)
b = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
print(b)
for switch, channel in zip(b, tx):
    print(f'switch = {switch} with channel = {channel}')

tensor([[[[-4., -4.],
          [ 3., -3.]],

         [[ 1., -3.],
          [ 3., -5.]],

         [[ 3.,  4.],
          [ 3., -4.]]],


        [[[-5.,  3.],
          [ 2.,  0.]],

         [[-2.,  3.],
          [ 1.,  2.]],

         [[-5., -2.],
          [ 3.,  3.]]],


        [[[ 1., -5.],
          [ 3.,  0.]],

         [[-4.,  4.],
          [-4.,  3.]],

         [[-3.,  1.],
          [ 1.,  4.]]],


        [[[-3.,  0.],
          [-1.,  0.]],

         [[-5., -4.],
          [ 1.,  0.]],

         [[-2.,  2.],
          [ 4., -1.]]]])
torch.Size([4])
tensor([0., 0., 0., 0.])
switch = 0.0 with channel = tensor([[[-4., -4.],
         [ 3., -3.]],

        [[ 1., -3.],
         [ 3., -5.]],

        [[ 3.,  4.],
         [ 3., -4.]]])
switch = 0.0 with channel = tensor([[[-5.,  3.],
         [ 2.,  0.]],

        [[-2.,  3.],
         [ 1.,  2.]],

        [[-5., -2.],
         [ 3.,  3.]]])
switch = 0.0 with channel = tensor([[[ 1., -5.],
         [ 3.,  0.]],

        

## 확률 식으로 sin, cos 사용

In [22]:
class BwithSinCos(pl.LightningModule):
    def __init__(self):
        super(BwithSinCos, self).__init__()
        
    def forward(self, x):
        p = -torch.sin(x) * torch.cos(x + (torch.pi/2))
        print('out of 1')
        print((p > 1).nonzero(as_tuple=True))
        print(p[(p > 1).nonzero(as_tuple=True)])
        print('out of 0')
        print((p < 0 ).nonzero(as_tuple=True))
        print(p[(p < 0 ).nonzero(as_tuple=True)])
        print()
        epsilon = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        return x * epsilon.to(self.device)

In [23]:
sc = BwithSinCos()
sc(x)

out of 1
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])
out of 0
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])



tensor([[ 0.,  0.,  7.],
        [ 0.,  0.,  0.],
        [ 0., 14.,  0.]])

In [24]:
i = torch.tensor([[1.56, torch.pi/2],
                  [0, 1]])

sc(i)

out of 1
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])
out of 0
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])



tensor([[1.5600, 1.5708],
        [0.0000, 0.0000]])

## 얽힘 (구현 방법 생각 중)

모든 칸을 돌면서 자기 주위의 1 개수를 센 다음 % 2 한 값을 자기 위치에 갖는다.

In [25]:
b = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
o = torch.zeros_like(b)
print(b)
print(o)

tensor([[1., 0., 1.],
        [0., 1., 1.],
        [0., 1., 1.]])
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


In [26]:
a = torch.randint(0, 2, (3, 4, 5))
a

tensor([[[0, 0, 0, 0, 1],
         [1, 0, 0, 1, 0],
         [1, 0, 1, 0, 1],
         [1, 0, 1, 1, 0]],

        [[1, 0, 1, 1, 1],
         [1, 0, 1, 1, 0],
         [0, 0, 1, 1, 0],
         [1, 1, 0, 0, 0]],

        [[0, 1, 1, 1, 0],
         [0, 1, 1, 1, 1],
         [1, 1, 0, 1, 1],
         [1, 0, 1, 1, 1]]])

In [27]:
(a==1).nonzero(as_tuple=True)
a[(a==1).nonzero(as_tuple=True)] += 1 

In [28]:
(a==1).nonzero(as_tuple=True)

(tensor([], dtype=torch.int64),
 tensor([], dtype=torch.int64),
 tensor([], dtype=torch.int64))

In [29]:
a

tensor([[[0, 0, 0, 0, 2],
         [2, 0, 0, 2, 0],
         [2, 0, 2, 0, 2],
         [2, 0, 2, 2, 0]],

        [[2, 0, 2, 2, 2],
         [2, 0, 2, 2, 0],
         [0, 0, 2, 2, 0],
         [2, 2, 0, 0, 0]],

        [[0, 2, 2, 2, 0],
         [0, 2, 2, 2, 2],
         [2, 2, 0, 2, 2],
         [2, 0, 2, 2, 2]]])

In [30]:
ones = (b==1).nonzero(as_tuple=False)

for i in ones:
    print(i)
    print(b[tuple(i)])

tensor([0, 0])
tensor(1.)
tensor([0, 2])
tensor(1.)
tensor([1, 1])
tensor(1.)
tensor([1, 2])
tensor(1.)
tensor([2, 1])
tensor(1.)
tensor([2, 2])
tensor(1.)


-------------------------------------

In [31]:
f = torch.distributions.bernoulli.Bernoulli(logits=alpha * x)

In [32]:
f.probs

tensor([[0.7109, 0.5000, 0.6682],
        [0.5000, 0.5000, 0.5000],
        [0.5000, 0.8022, 0.5000]])

In [33]:
e = torch.exp(alpha * x) / (torch.exp(alpha * x) + 1)
print(e)

tensor([[0.7109, 0.5000, 0.6682],
        [0.5000, 0.5000, 0.5000],
        [0.5000, 0.8022, 0.5000]])


In [34]:
p = -torch.sin(x) * torch.cos(x+ (torch.pi/2))
print(p)
torch.distributions.bernoulli.Bernoulli(probs=p).sample()

tensor([[0.1698, 0.0000, 0.4316],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.9813, 0.0000]])


tensor([[0., 0., 1.],
        [0., 0., 0.],
        [0., 1., 0.]])